# Seleção de participantes - Gênero (female x male)

Subamostra pareada por sexo usada nas análises `Gender/Filtered` (Gordon e Glasser).

**Regra de pareamento**

* Âncoras: `sex == 'male'` (homens), percorridas na ordem da planilha.
* Para cada âncora, candidatos com `sex == 'female'` (mulheres) e mesmos `religious_now`, `handedness`, `education_level`.
* Escolhe-se o candidato com menor `|Δage| + |Δbackground_SES|` (coluna `dist`), sem reposição.
* Resultado: 380 homens + 380 mulheres = 760.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats

# Planilha com os 928 participantes do ID1000 (participants.tsv do OpenNeuro ds003097 + colunas usadas no projeto)
PARTICIPANTS_CSV = "https://docs.google.com/spreadsheets/d/13n8f5tILuIVF6Cxi2izX020_fYNjNNZk/export?format=csv"

# Pasta local de dados do projeto (a que contém ID1000Files/). Rode o notebook a partir dela ou ajuste aqui.
DATA_ROOT = "."
OUTPUT_DIR = DATA_ROOT

OUTPUT_XLSX = os.path.join(OUTPUT_DIR, "selected_participantsGender.xlsx")

df = pd.read_csv(PARTICIPANTS_CSV)
df.shape

(928, 31)

Identificar sujeitos que não têm funcional

In [2]:
# Remove sujeitos que têm anat/ mas não têm func/ (sem fMRI funcional): 928 -> 881
base_dir = os.path.join(DATA_ROOT, "ID1000Files", "raw")

anat_only = []
for dataset in os.listdir(base_dir):
    dataset_dir = os.path.join(base_dir, dataset)
    if not os.path.isdir(dataset_dir):
        continue
    subs = [d for d in os.listdir(dataset_dir) if d.startswith("sub-") and os.path.isdir(os.path.join(dataset_dir, d))]
    for subj in subs:
        anat_dir = os.path.join(dataset_dir, subj, "anat")
        func_dir = os.path.join(dataset_dir, subj, "func")
        has_anat = os.path.isdir(anat_dir) and any(os.path.isfile(os.path.join(anat_dir, f)) for f in os.listdir(anat_dir))
        has_func = os.path.isdir(func_dir) and any(os.path.isfile(os.path.join(func_dir, f)) for f in os.listdir(func_dir))
        if has_anat and not has_func:
            anat_only.append(subj)

df = df[~df["participant_id"].isin(anat_only)]
print("sujeitos sem func:", len(anat_only), "| pool:", len(df))

sujeitos sem func: 47 | pool: 881


## Pareamento

In [3]:
# Pareamento 1:1 guloso, sem reposição
pool = df   # todos os 881 participantes com fMRI
ancoras = pool[pool["sex"] == "male"].copy()      # homens
candidatos_pool = pool[pool["sex"] == "female"].copy()   # mulheres

EXATAS = ['religious_now', 'handedness', 'education_level']   # variáveis pareadas exatamente

pares = []
for idx, row in ancoras.iterrows():
    candidatos = candidatos_pool
    for col in EXATAS:
        candidatos = candidatos[candidatos[col] == row[col]]
    if candidatos.empty:
        continue
    candidatos = candidatos.copy()

    # Distância combinando idade e SES
    candidatos["dist"] = (
        (candidatos["age"] - row["age"]).abs() +
        (candidatos["background_SES"] - row["background_SES"]).abs()
    )
    mais_proximo = candidatos.sort_values("dist").iloc[0]

    pares.append(row)
    pares.append(mais_proximo)

    # Remove o já emparelhado
    candidatos_pool = candidatos_pool.drop(mais_proximo.name)

df_pares = pd.DataFrame(pares).reset_index(drop=True)
df_final = df_pares.sort_values(by=["age"])
print(df_final["sex"].value_counts())
df_final

sex
male      380
female    380
Name: count, dtype: int64


,participant_id,age,sex,handedness,BMI,education_level,background_SES,IST_fluid,IST_memory,IST_crystallised,...,sexual_attraction_F,gender_identity_M,gender_identity_F,religious_upbringing,religious_now,religious_importance,DWI_TR_run1,DWI_TR_run2,DWI_TR_run3,dist
76,sub-0716,19.00,male,right,27,medium,3.0,82.0,41.0,13.0,...,7.0,7.0,1.0,no,yes,2.0,6.374,6.374,6.374,NaN
510,sub-0475,19.75,male,right,22,high,5.0,104.0,57.0,37.0,...,7.0,7.0,1.0,no,no,NaN,6.312,6.312,6.312,NaN
759,sub-0215,20.00,female,left,24,high,2.0,122.0,45.0,35.0,...,NaN,NaN,NaN,no,no,NaN,6.313,6.313,6.313,3.00
52,sub-0282,20.00,male,right,17,low,4.0,24.0,28.0,16.0,...,7.0,7.0,1.0,yes,yes,2.0,6.311,6.311,6.311,NaN
575,sub-0379,20.00,female,right,30,high,4.0,131.0,62.0,45.0,...,NaN,NaN,NaN,yes,no,NaN,6.374,6.374,6.374,0.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
528,sub-0500,26.00,male,right,23,high,3.5,102.0,55.0,42.0,...,7.0,2.0,1.0,no,no,NaN,6.731,6.311,6.311,NaN
636,sub-0639,26.00,male,right,24,high,4.0,104.0,60.0,34.0,...,NaN,NaN,NaN,no,no,NaN,6.313,6.313,6.313,NaN
529,sub-0503,26.00,female,right,21,high,3.5,91.0,50.0,26.0,...,NaN,NaN,NaN,no,no,NaN,6.374,6.374,6.374,0.00
622,sub-0625,26.00,male,right,25,medium,6.0,136.0,62.0,47.0,...,2.0,7.0,1.0,no,no,NaN,6.838,6.838,6.838,NaN


## Verificação do balanço

In [4]:
# Balanço entre os grupos pareados
g1 = df_final[df_final["sex"] == "male"]
g2 = df_final[df_final["sex"] == "female"]
for var in ["age", "background_SES", "IST_intelligence_total"]:
    t, p = stats.ttest_ind(g1[var].dropna(), g2[var].dropna(), equal_var=True)
    print(f"{var:24s} male={g1[var].mean():.2f}  female={g2[var].mean():.2f}  t={t:.3f}  p={p:.3f}")
for var in EXATAS:
    print(var, "idêntico entre grupos:", (g1[var].value_counts().sort_index() == g2[var].value_counts().sort_index()).all())

age                      male=22.82  female=22.83  t=-0.106  p=0.915
background_SES           male=4.08  female=3.96  t=1.422  p=0.155
IST_intelligence_total   male=207.19  female=193.72  t=4.586  p=0.000
religious_now idêntico entre grupos: True
handedness idêntico entre grupos: True
education_level idêntico entre grupos: True


## Gravação

In [6]:
df_final.to_excel(OUTPUT_XLSX, index=False)